# Reading another project's results

`labdata` lets one project read result files produced by another, without
submodules, without cloning, and without downloading anything by hand.

A repository publishes by committing a `labdata.yml` in the `results/`
directory at its root, naming the files it wants others to see and saying what
each one holds. Everything else in `results/` stays private to that project.

Repositories are read wherever they are: clones on this machine, clones on a
server reached over ssh, and repositories on GitHub read over the API without
being cloned at all.

In [1]:
import labdata
import vscodenb
import pandas as pd
from pandas.api.types import is_object_dtype

class nice:
    def __rlshift__(self, df):
        "Left align columns of params frame: df << nice()"
        s = df.style
        s.set_table_styles(
            {c: [{'selector': '', 'props': [('text-align', 'left')]}] 
                 for c in df.columns if is_object_dtype(df[c])},
            overwrite=False
        )
        display(s)  

In [ ]:
cfg = labdata.config.Config(
    # search these paths under each repo roots for labdata.yml files:
    labdata_dirs=['results'], 

    # # search all github repos under these accounts:
    # owners=["munch-group", ], 
    
    # folders to search for working copies of git repos:
    roots = ["kmt@login.genome.au.dk:xy-drive/people/kmt", ]
)
labdata.use_config(cfg)
labdata.refresh()

scanning clones:   0%|          | 0/12 [00:00<?, ?repo/s]

reading GitHub:   0%|          | 0/156 [00:00<?, ?repo/s]

KeyboardInterrupt: 

### `labdata.yml`

To register files or directories as data available to `labdata`, add a `labdata.yml` to the repository `results`. Keys are file names, paths or globs and values say what the file holds. You can list parquet folders as if they were files, since they load as such.

```yml
files:
  hits.csv: Genome-wide association hits, p < 5e-8
  qc/summary.csv: Per-sample genotyping quality
  by_chrom: Per-chromosome effect sizes, one file per chromosome
```

A key beginning with `/` is a path from the repository root rather than from the
`labdata.yml`, which publishes a file that is kept elsewhere in the repository.
Any tracked file can be named this way; `..` cannot be used to climb out.

```yml
files:
  hits.csv: In the results directory, as usual
  /data/reference/samples.csv: Somewhere else in the repository
  /data/raw/*.tsv: A pattern, matched against the path from the root
```


### Files too large to commit

A file the pipeline writes but nobody wants in git history is published as a
**link**: a symbolic link committed in `results/`, pointing at wherever the
pipeline put it, and a *stamp* in the `labdata.yml` saying which content that
link stands for.

```console
$ ln -s ../steps/very_large_file.csv results/very_large_file.csv
$ labdata stamp
  stamped  results/very_large_file.csv  (41M)
stamped 1 file; commit the labdata.yml to publish this version
```

```yml
files:
  very_large_file.csv:
    description: Merged per-sample table
    sha256: "3f9a...c1"        # written by `labdata stamp`
    size: 41231234
```

Git versions the link, not the bytes behind it, so a link on its own would let
the content change without the version changing. The stamp closes that gap: it
is committed, so **the commit that changes a stamp is the new version**, and
`labdata.versions()` lists the commits in which the stamp moved. Regenerating
the file means running `labdata stamp` again and committing that;
`labdata stamp --check` writes nothing and exits non-zero when a stamp is out
of date, for a pre-commit hook or CI.

Nothing is copied. When the file and the cache are on one filesystem,
`labdata.get()` returns a **hard link** to the file the pipeline wrote:

```python
os.stat(labdata.get("proj", "very_large_file.csv")).st_ino
os.stat("steps/very_large_file.csv").st_ino          # the same inode
```

What it costs is that the bytes exist only where the pipeline wrote them.
Cloning the repository from GitHub gets you the link and no content, and
`labdata.get()` says so, naming the file and the machine; reading it means
configuring a root that has it, including one on a server over ssh.


## Config

Roots on another machine are read over ssh. Where the host asks for a key
passphrase or a second factor, ssh prompts for it at the terminal, once, and the
shared connection covers the commands that follow. In a notebook the prompt opens at the top of the
window, as any `getpass` in a cell does, and what you type goes to ssh. To be
asked once a day rather than once a session, give the host a
`ControlMaster`/`ControlPersist` block in `~/.ssh/config`; labdata reads through
the connection you already opened.

`labdata` configuration is read from a config file that can be generated with `labdata config --init` and shown using:

In [2]:
labdata.active_config()

Config(roots=[],
       owners=['munch-group'],
       repos=[],
       labdata_dirs=['results', 'data'],
       include=['*.csv',
                '*.tsv',
                '*.txt',
                '*.parquet',
                '*.pq',
                '*.h5',
                '*.hdf',
                '*.hdf5',
                '*.store',
                '*.json',
                '*.jsonl',
                '*.xlsx',
                '*.bed',
                '*.gff',
                '*.vcf',
                '*.vcf.gz',
                '*.pkl',
                '*.pickle',
                '*.npy',
                '*.npz',
                '*.feather',
                '*.zarr'],
       exclude=['*.png',
                '*.pdf',
                '*.svg',
                '*.html',
                '*.md',
                '.gitkeep',
                '*.log'],
       min_bytes=0,
       max_bytes=0)

You can also generate one on the fly in the notebook and pass along to each labdata function:

In [ ]:
cfg = labdata.config.Config(
    # search these paths under each repo roots for labdata.yml files:
    labdata_dirs=['results', 'steps/data'], 
    # only include these file types among those listed in labdata files (leave empty to include all)
    include=['*.parquet', '*.csv', '*.tsv'],
    # exclude include these file types in when listed in labdata files:
    exclude=['.store', '*.fa', '*.vcf', '*.bim', '*.fam', '*.ped', '*.bam'],

    # search all github repos under these accounts:
    owners=["munch-group", "erikfogh" ], 
    # search these github repos:
    repos=['munch-group/relate1Kgenomes', 'munch-group/atlas-variant-ages'], 
    
    # folders to search for working copies of git repos:
    roots = ["~/Documents/projects", "kmt@login.genome.au.dk:xy-drive/people/kmt", ]
)
labdata.use_config(cfg)
labdata.active_config() 

Config(roots=['~/Documents/projects',
              'kmt@login.genome.au.dk:xy-drive/people/kmt'],
       owners=['munch-group', 'erikfogh'],
       repos=['munch-group/relate1Kgenomes', 'munch-group/atlas-variant-ages'],
       labdata_dirs=['results', 'steps/data'],
       include=[],
       exclude=[],
       min_bytes=0,
       max_bytes=0)

In [24]:
labdata.use_config(None)  # back to ~/.config/labdata/config.toml

Config(roots=[],
       owners=['munch-group'],
       repos=[],
       labdata_dirs=['results', 'data'],
       include=['*.csv',
                '*.tsv',
                '*.txt',
                '*.parquet',
                '*.pq',
                '*.h5',
                '*.hdf',
                '*.hdf5',
                '*.store',
                '*.json',
                '*.jsonl',
                '*.xlsx',
                '*.bed',
                '*.gff',
                '*.vcf',
                '*.vcf.gz',
                '*.pkl',
                '*.pickle',
                '*.npy',
                '*.npz',
                '*.feather',
                '*.zarr'],
       exclude=['*.png',
                '*.pdf',
                '*.svg',
                '*.html',
                '*.md',
                '.gitkeep',
                '*.log'],
       min_bytes=0,
       max_bytes=0)

## Refresh

In [4]:
labdata.refresh()

scanning clones:   0%|          | 0/16 [00:00<?, ?repo/s]

KeyboardInterrupt: 

## Repos

In [ ]:
labdata.repos()

,repo,files,bytes,latest
0,munch-group/relate1Kgenomes,1,41270846,2026-08-31
1,munch-group/tree-stats,1,19,2026-08-30


## List

`labdata.list()` is the python side of `labdata list`.

In [27]:
labdata.list() << nice()

,owner,repo,name,description,date,github,path,dir,bytes,tags,lfs
0,munch-group,CTCF-inversion-data,ctcf_encode_files.tsv,blah,2026-09-02,munch-group/CTCF-inversion-data,results/ctcf_encode_files.tsv,results,115265,,False
1,munch-group,CTCF-inversion-data,ctcf_sites_hg38.parquet,blah,2026-09-02,munch-group/CTCF-inversion-data,results/ctcf_sites_hg38.parquet,results,11741434,,False
2,munch-group,CTCF-inversion-data,ctcf_sites_hg38_chm13.parquet,blah,2026-09-02,munch-group/CTCF-inversion-data,results/ctcf_sites_hg38_chm13.parquet,results,17454688,,False
3,munch-group,relate1Kgenomes,relate_snps_p_vals.parquet,Genome SNPs (hg38) with a log p-value below -2w,2026-09-02,munch-group/relate1Kgenomes,results/relate_snps_p_vals.parquet,results,82477487,,False
4,munch-group,tree-stats,dummy.csv,Some dummy csv file,2026-08-30,munch-group/tree-stats,results/dummy.csv,results,19,,False


In [28]:
labdata.list(brief=True) << nice()

,owner,repo,name,size,description,date
0,munch-group,CTCF-inversion-data,ctcf_encode_files.tsv,0.1 MB,blah,2026-09-02
1,munch-group,CTCF-inversion-data,ctcf_sites_hg38.parquet,11.7 MB,blah,2026-09-02
2,munch-group,CTCF-inversion-data,ctcf_sites_hg38_chm13.parquet,17.5 MB,blah,2026-09-02
3,munch-group,relate1Kgenomes,relate_snps_p_vals.parquet,82.5 MB,Genome SNPs (hg38) with a log p-value below -2w,2026-09-02
4,munch-group,tree-stats,dummy.csv,0.0 MB,Some dummy csv file,2026-08-30


In [29]:
labdata.list(version=True) << nice()

,owner,repo,name,description,date,github,path,dir,bytes,tags,lfs,version
0,munch-group,CTCF-inversion-data,ctcf_encode_files.tsv,blah,2026-09-02,munch-group/CTCF-inversion-data,results/ctcf_encode_files.tsv,results,115265,,False,14ebf0da170dfe4c0baca2840ff8d3be81640b9c
1,munch-group,CTCF-inversion-data,ctcf_sites_hg38.parquet,blah,2026-09-02,munch-group/CTCF-inversion-data,results/ctcf_sites_hg38.parquet,results,11741434,,False,14ebf0da170dfe4c0baca2840ff8d3be81640b9c
2,munch-group,CTCF-inversion-data,ctcf_sites_hg38_chm13.parquet,blah,2026-09-02,munch-group/CTCF-inversion-data,results/ctcf_sites_hg38_chm13.parquet,results,17454688,,False,14ebf0da170dfe4c0baca2840ff8d3be81640b9c
3,munch-group,relate1Kgenomes,relate_snps_p_vals.parquet,Genome SNPs (hg38) with a log p-value below -2w,2026-09-02,munch-group/relate1Kgenomes,results/relate_snps_p_vals.parquet,results,82477487,,False,590c4f17d5a2ff765e61b6df9a1fa88318c9760d
4,munch-group,tree-stats,dummy.csv,Some dummy csv file,2026-08-30,munch-group/tree-stats,results/dummy.csv,results,19,,False,770170789d73428efb0193d4cf04a353587db9cd


## Get

In [33]:
tmp_path = labdata.get("atlas-variant-ages", "atlas_variant_ages.parquet")

Add version="619b60e3af78bed6494222f82f754a3863249fe9" to pin this version.


In [9]:
df = pd.read_parquet(tmp_path)
df.head()

,chrom,pop,pos,p_half_freq,p_two_alleles
0,chr1,ACB,817341,-0.473037,-2.88930
1,chr1,ACB,892733,-1.231820,-2.30420
2,chr1,ACB,893360,-1.231820,-2.30420
3,chr1,ACB,897538,-1.319140,-4.10702
4,chr1,ACB,901516,-1.319140,-3.19570


In [10]:
filters = [
    ('pos', '>=', 892733), 
    ('pos', '<', 901516),
    ('pop', '==', "ACB"),
    ('chrom', '==', "chrX"),
]
pd.read_parquet(tmp_path, filters=filters)

,chrom,pop,pos,p_half_freq,p_two_alleles
0,chrX,ACB,893285,-1.273310,-4.51818
1,chrX,ACB,895075,-1.713310,-4.48943
2,chrX,ACB,895885,-1.274320,-2.17077
3,chrX,ACB,896099,-1.274320,-2.17077
4,chrX,ACB,896561,-1.713310,-4.48943
5,chrX,ACB,897491,-1.355870,-5.26528
6,chrX,ACB,897556,-0.750727,-3.91539
7,chrX,ACB,897658,-0.574665,-2.00685
8,chrX,ACB,898415,-0.301048,-4.08054
9,chrX,ACB,899070,-0.574665,-2.00685


In [ ]:
tmp_path = labdata.get("munch-group/relate1Kgenomes", "results/relate_snps_p_vals.parquet")

Add version="590c4f17d5a2ff765e61b6df9a1fa88318c9760d" to pin this version.


In [ ]:
df = pd.read_parquet(tmp_path)
df.head()

,variant_id,chrom,pos,ref,alt,anc,age_mode,age_lo95ci,age_hi95ci
0,rs537182016,1,10539,C,A,.,527.489,17.852,1118.33
1,rs558604819,1,10642,G,A,.,11418.100,10038.000,12847.50
2,rs575272151,1,11008,C,G,.,16174.800,14789.800,17615.10
3,rs544419019,1,11012,C,G,.,20599.300,18875.200,22362.90
4,rs561109771,1,11063,T,G,.,5802.930,4794.360,6848.18


In [16]:
# silent, because this is still the current version
tmp_path = labdata.get("munch-group/relate1Kgenomes", 
                       "results/relate_snps_p_vals.parquet",
                       version="590c4f17d5a2ff765e61b6df9a1fa88318c9760d")

The file name is enough when it is unambiguous; otherwise give as much of the
path as it takes.

In [17]:
labdata.get("relate1Kgenomes", "relate_snps_p_vals.parquet")

Add version="590c4f17d5a2ff765e61b6df9a1fa88318c9760d" to pin this version.


PosixPath('/Users/kmt/.cache/labdata/files/munch-group__relate1Kgenomes/590c4f17d5a2ff765e61b6df9a1fa88318c9760d/results/relate_snps_p_vals.parquet')

## History

The catalog stamps every file with the repository's current commit.
`labdata.versions()` shows the commits in which the file itself changed, which
is the useful set to pin.

In [18]:
labdata.versions("relate1Kgenomes", "relate_snps_p_vals.parquet")

,version,date,bytes,parts,tags,subject
0,f4f1fc621cb9dfac3610f7f47d90caf640b43571,2026-09-01,82477487,3,,Added parquet fixed data


## When nothing shows up

An empty catalog has several ordinary causes and they look alike from outside.
`labdata.diagnose()` says which it is, reading only local git.

In [14]:
print("\n".join(labdata.diagnose(labdata.Config(roots=["~/no-such-place"]))))

  ~/no-such-place: no such directory
  github: no owners or repos configured, so nothing is read from GitHub
